In [1]:
import os

# Must run BEFORE any `import torch` in this kernel (CUDA_VISIBLE_DEVICES is read once).
# Changing GPU_IDS or PILOT_MODE requires restarting the kernel.
GPU_IDS = "4,5"  # physical GPU ids dedicated to this session, e.g. "0" or "4,5"
PILOT_MODE = "small_1gpu_pair"  # "small_1gpu_pair" (Qwen-3B + Gemma-1B side by side) or "large_2gpu_single"
LARGE_MODEL_KEY = "gemma-3-4b-it"  # used only when PILOT_MODE == "large_2gpu_single": "qwen2.5-7b-instruct" or "gemma-3-4b-it"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS

In [2]:
import os
import sys
from pprint import pprint

import torch

from scripts.local_qa_generation import (  # noqa: E402
    MODEL_REGISTRY,
    PREGUNTAS_COMUNES,
    get_available_devices,
    load_local_model,
    process_full_dataset_local,
    process_single_context_local,
)

/var/lib/datausers_jupyterhub/lfalconi_storage/miniforge3/envs/pyt-eqa-fge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv
from huggingface_hub import login

# Reads HUGGINGFACE_TOKEN from a .env file at the repo root (never commit the token).
load_dotenv()
hf_token = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    raise RuntimeError(
        "HUGGINGFACE_TOKEN not found. Add it to a .env file at the repo root."
    )
login(hf_token)

In [4]:
from datasets import load_dataset

MIN_WORDS = 82
MAX_WORDS = 150

dataset_path = "LeninGF/autotrain-data-robberyclassification"
ds = load_dataset(dataset_path)


def count_words(sample):
    sample["word_count"] = len(sample["relato"].split())
    return sample


ds = ds.map(count_words, batched=False)
filtered_ds = ds["train"].filter(
    lambda batch: [MIN_WORDS <= x <= MAX_WORDS for x in batch["word_count"]],
    batched=True,
    batch_size=1000,
)
filtered_ds

Dataset({
    features: ['relato', 'labels', 'word_count'],
    num_rows: 174594
})

In [5]:
PILOT_SIZE = 50
pilot_indices = list(range(min(PILOT_SIZE, len(filtered_ds))))
pilot_ds = filtered_ds.select(pilot_indices)


def pilot_context_id(i):
    """Map pilot_ds row i back to its original filtered_ds index."""
    return f"context_{pilot_indices[i]}"


pilot_ds

Dataset({
    features: ['relato', 'labels', 'word_count'],
    num_rows: 50
})

In [10]:
%%time
import os
import traceback
import torch

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.device_count() =", torch.cuda.device_count())
print("available devices =", get_available_devices())

PILOT_SIZE = 1
pilot_indices = list(range(min(PILOT_SIZE, len(filtered_ds))))
pilot_ds = filtered_ds.select(pilot_indices)

# Si quieres probar Qwen:
model_name = "qwen2.5-3b-instruct"
model = load_local_model("qwen2.5-3b-instruct", gpu_ids=[0], quantize_4bit=True)

ctx = pilot_ds[0]["relato"]
q = PREGUNTAS_COMUNES[2]
# q = PREGUNTAS_COMUNES

print("Contexto preview:", ctx[:500])
print("Pregunta:", q)

try:
    result = process_single_context_local(
        ctx,
        [q],
        model,
        context_id="debug_qwen",
        model_name=model_name
    )
    print("RESULTADO:", result)
except Exception:
    traceback.print_exc()

CUDA_VISIBLE_DEVICES = 4,5
torch.cuda.device_count() = 2
available devices = ['cuda:0', 'cuda:1']
Loading Qwen/Qwen2.5-3B-Instruct on logical gpu ids [0] ...


Loading weights: 100%|██████████| 434/434 [00:02<00:00, 155.67it/s]


Contexto preview: es el caso señor fiscal que el día 22 de octubre del 2014 a las 20h40 aproximadamente en circunstancias en que deje estacionado mi camioneta de placas gmd0996 color blanco marca chevrolet de mi propiedad en la dirección antes mencionada al momento de ir por ella me encuentro con la novedad que no estaba se la habían robado en el interior de la camioneta ahí una arma de fuego revolver calibre 38 de fabricación nacional que pertenece a compañía armiled por lo expuesto señor fiscal solicito que se 
Pregunta: ¿A qué hora sucedió el robo?
RESULTADO: [{'context': 'es el caso señor fiscal que el día 22 de octubre del 2014 a las 20h40 aproximadamente en circunstancias en que deje estacionado mi camioneta de placas gmd0996 color blanco marca chevrolet de mi propiedad en la dirección antes mencionada al momento de ir por ella me encuentro con la novedad que no estaba se la habían robado en el interior de la camioneta ahí una arma de fuego revolver calibre 38 de fabricación naci

In [ ]:
%%time
model_name = "gemma-3-1b-it"
model2 = load_local_model(model_name, gpu_ids=[1], quantize_4bit=True)

try:
    result2 = process_single_context_local(
        ctx,
        [q],
        model2,
        context_id="debug_gemma",
        model_name=model_name
    )
    print("RESULTADO GEMMA:", result2)
except Exception:
    traceback.print_exc()

Loading google/gemma-3-1b-it on logical gpu ids [1] ...


Loading weights: 100%|██████████| 340/340 [00:00<00:00, 379.30it/s]
